<a href="https://colab.research.google.com/github/Zain506/MedCLIP-SAM/blob/main/notebooks/ImageSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Segmentation

**MedCLIP-SAM**
[Paper](https://arxiv.org/pdf/2403.20253)


**gScoreCAM**
[Paper](https://www.google.com/url?q=https%3A%2F%2Fopenaccess.thecvf.com%2Fcontent%2FACCV2022%2Fpapers%2FChen_gScoreCAM_What_objects_is_CLIP_looking_at_ACCV_2022_paper.pdf)

In [72]:
%pip install open-clip-torch -q

In [73]:
from google.colab import drive

drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


## Load BiomedCLIP

In [74]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_path = "/content/drive/MyDrive/colab/MedCLIP-SAM/biomedclip_weights.pth"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device)

In [75]:
# We are looking for the final attention layer
# We place a hook in it to retrieve
modules = dict(model.named_modules())
print(modules["visual"].transformer.resblocks[10].attn) # For full model, print(modules[""])

MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
)


# Define a method to view the activations and gradients

In [86]:
def online_test(img, text):
    activations = {}

    def forward_hook(module, inp, out):
        out = out[0]
        out.retain_grad() # Save gradient
        activations["attn"] = out # Store

    layer = model.visual.transformer.resblocks[-2].attn # 2nd last attention layer
    handle_fwd = layer.register_forward_hook(forward_hook) # Register hook

    # forward
    img_emb = model.encode_image(img)
    text_emb = model.encode_text(text)

    # differentiable loss
    sim = img_emb @ text_emb.T
    loss = -sim.diag().mean() # Temporary loss function

    loss.backward() # Backward pass

    handle_fwd.remove() # Remove hook

    act = activations["attn"] # Retrieve activations
    grad = act.grad # Retrieve gradients

    return act, grad, loss


# Load data to test

In [77]:
from datasets import load_dataset
ds = load_dataset("adishourya/MEDPIX-ClinQA") # Same MedPIX dataset that fine-tuned MedCLIP
train_valid = ds["train"].train_test_split(test_size=0.1)
training = train_valid["train"].select(range(100))
test = train_valid["test"]

In [78]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def getData(i): # Retrieve data and generate initial embeddings
  text = tokenizer(training[i]["question"] + "\n" + training[0]["answer"])
  image = preprocess(training[i]["image_id"]).unsqueeze(0).to(device)

  return text, image


In [88]:
text, image = getData(0)
tmp = online_test(image, text)

In [102]:
print("Activations".center(50, " "))
obj = tmp[0][0]
print(obj.shape)
print(obj)
print("\n"*2)
print("Gradients".center(50, " "))
obj2 = tmp[1][0]
print(obj2.shape)
print(obj2)
print("\n"*2)
print("Loss".center(50, " "))
print("\n")
print(tmp[2])

                   Activations                    
torch.Size([50, 768])
tensor([[ 0.0419, -0.0297, -0.1904,  ...,  0.0982,  0.1381,  0.0556],
        [ 0.2093, -0.0734, -0.2608,  ..., -0.0877,  0.1582,  0.1712],
        [ 0.2017, -0.0995, -0.2440,  ..., -0.0629,  0.1716,  0.1489],
        ...,
        [-0.0589,  0.0174, -0.1286,  ..., -0.0170, -0.0193,  0.1766],
        [ 0.1166, -0.0244, -0.2125,  ..., -0.0540,  0.1243,  0.1566],
        [ 0.0947, -0.0047, -0.2153,  ..., -0.0333,  0.1083,  0.1488]],
       grad_fn=<SelectBackward0>)



                    Gradients                     
torch.Size([50, 768])
tensor([[ 1.6599e-02, -2.0468e-02,  1.3135e-01,  ..., -5.5356e-02,
         -1.8153e-01, -2.6640e-01],
        [ 3.9371e-04, -2.5132e-04, -5.5410e-04,  ...,  1.0049e-04,
         -7.2924e-04, -3.2166e-05],
        [ 5.9077e-04, -2.7126e-04, -5.5348e-04,  ...,  9.8575e-05,
         -1.0053e-03, -1.0783e-04],
        ...,
        [-5.4884e-04, -5.5775e-04, -1.0175e-04,  ..., -1.2564